# Pyramid 3D CNN (FPN-style) — 96^3 OCT, Kaggle GPU

Experiment for `models.glaucoma.model.Pyramid3DCNN`: 4-level 3D feature pyramid, anisotropic
`(2,2,1)` pooling (full A-scan depth), GroupNorm, residual stages, top-down lateral fusion and a
multi-scale pooled head.

**Resolution choice:** raw 200^3 measured **33 s/optimizer-step** on a T4 (2.4 h/epoch, ~36 h for
15 epochs) — impossible inside Kaggle's session limit. The run therefore uses the verified
**Bilateral-96** storage (bit-identical to on-the-fly resize of Bilateral-200, approved for the 3D
branch), ~4 h for 15 epochs, same batch/lr/epoch as the 96^3 single-branch sweep.

* **Data**: `tqhuyen/harvard-oct-glaucoma-200-bilateral-96` — 2100/300/900, uint8
  `(n, 1, 96, 96, 96)`, ~2.9 GB.
* **Reference** (`docs/3d-backbone-sweep.md`, same 96^3 group): `single3d-resxt3d`
  test acc 77.11%, AUC 85.38%, MCC 0.542.
* **W&B**: project `glaucoma-thesis`, run `pyramid-bilateral96-s42`.

## Kaggle setup
1. Notebook settings -> **Internet: On**.
2. Credentials: `kaggle_secrets` (HF_TOKEN, WANDB_API_KEY) or the private dataset
   `trnquanghuyn/glaucoma-secrets` (fallback already wired in).
3. Accelerator: **GPU T4** (or P100).

## Run policy
Real run is the default (`SMOKE = 0`). Set `SMOKE = 1` for a wiring check on synthetic 32^3 data.


In [ ]:
import os, sys, json, subprocess, shutil, time
from pathlib import Path

SMOKE = os.environ.get("PYRAMID_SMOKE", "0") == "1"   # <- 1 = wiring check, 0 = real 96^3 run

IN_KAGGLE = Path("/kaggle").is_dir()
REPO = Path("/kaggle/working/glaucoma-thesis") if IN_KAGGLE else Path.cwd()
REPO_URL = "https://github.com/Tqhuyen/glaucoma-thesis.git"
RUN = "pyramid-synth-smoke" if SMOKE else "pyramid-bilateral96-s42"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
WORK = (Path("/kaggle/working") if IN_KAGGLE else REPO / ".tmp_pyramid") / ("pyramid200_smoke" if SMOKE else "pyramid200")
DATA_DIR = (SCRATCH / "pyramid200_data") if IN_KAGGLE else WORK / "data"
OUT_DIR = WORK / "outputs"
os.environ.setdefault("HF_HOME", str(SCRATCH / "hf"))
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
print("SMOKE =", SMOKE, "| repo =", REPO, "| run =", RUN)
if IN_KAGGLE:
    free_gb = shutil.disk_usage(SCRATCH).free / 2**30
    print(f"scratch {SCRATCH} free: {free_gb:.1f} GB (need ~4 GB for bilateral-96)")
    assert free_gb > 4 or SMOKE, "not enough scratch disk for bilateral-96"


In [ ]:
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
        os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
        print("secrets: kaggle_secrets")
    except Exception as exc:
        candidates = list(Path("/kaggle/input").rglob("secrets.env"))
        assert candidates, (
            f"kaggle_secrets failed ({type(exc).__name__}); no secrets.env under /kaggle/input; "
            f"mounted={[p.name for p in Path('/kaggle/input').iterdir()]}"
        )
        for line in candidates[0].read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())
        print("secrets: dataset fallback", candidates[0])
else:
    from pipeline.utils import load_env_file
    load_env_file()
assert os.environ.get("HF_TOKEN") and os.environ.get("WANDB_API_KEY"), "missing HF_TOKEN / WANDB_API_KEY"
print("secrets loaded")


In [ ]:
if not (REPO / "pipeline" / "train.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
model_src = (REPO / "models" / "glaucoma" / "model.py").read_text()
assert "class Pyramid3DCNN" in model_src and "pyramid3dcnn" in model_src, "pyramid code missing from repo"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb", "huggingface_hub", "scikit-learn", "tensorboard"], check=True)
sys.path.insert(0, str(REPO))
print("repo ready:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import numpy as np

SPLITS = ("Training", "Validation", "Test")
HF_REPO = "tqhuyen/harvard-oct-glaucoma-200-bilateral-96"
HF_FILES = {s: f"{s}_volumes_dn96.npy" for s in ("Training", "Validation", "Test")}
RES = 32 if SMOKE else 96

DATA_DIR.mkdir(parents=True, exist_ok=True)
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in (("Training", 40), ("Validation", 8), ("Test", 8)):
        y = np.array([i % 2 for i in range(n)], dtype=np.int64)
        v = rng.integers(30, 70, size=(n, 1, RES, RES, RES), dtype=np.uint8)
        v[y == 1] += 120
        np.save(DATA_DIR / f"{split}_volumes.npy", v)
        np.save(DATA_DIR / f"{split}_labels.npy", y)
else:
    from huggingface_hub import hf_hub_download
    for split in SPLITS:
        vol = hf_hub_download(HF_REPO, HF_FILES[split], repo_type="dataset", token=os.environ["HF_TOKEN"])
        lab = hf_hub_download(HF_REPO, f"{split}_labels.npy", repo_type="dataset", token=os.environ["HF_TOKEN"])
        for src, name in ((vol, f"{split}_volumes.npy"), (lab, f"{split}_labels.npy")):
            dst = DATA_DIR / name
            if not dst.exists():
                try:
                    dst.symlink_to(src)
                except OSError:
                    shutil.copyfile(src, dst)
for split in SPLITS:
    v = np.load(DATA_DIR / f"{split}_volumes.npy", mmap_mode="r")
    l = np.load(DATA_DIR / f"{split}_labels.npy")
    expected = (1, RES, RES, RES)
    assert len(v) == len(l) and tuple(v.shape[1:]) == expected, (v.shape, l.shape)
    sample = np.asarray(v[0], dtype=np.float32)
    print(split, v.shape, v.dtype, "class counts", np.bincount(l).tolist(),
          "min/max", float(sample.min()), float(sample.max()))


In [ ]:
import torch
from models.glaucoma.model import Pyramid3DCNN

assert torch.cuda.is_available(), "enable a Kaggle GPU accelerator"
m = Pyramid3DCNN().eval().cuda()
with torch.no_grad():
    out = m(torch.randn(1, 1, RES, RES, RES, device="cuda"))
print("params M:", round(sum(p.numel() for p in m.parameters()) / 1e6, 3))
print("logits:", tuple(out["logits"].shape), "| peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))
assert out["logits"].shape == (1, 2)
del m, out
torch.cuda.empty_cache()

### Training configuration
`fuse_levels=3` fuses levels at 50^3/25^3/12^3 (H/W), depth stays 200. If the T4/P100 OOMs,
rerun with `model.fuse_levels=2` (drops the 50x50x200 lateral) or batch 1 with grad-accum 8.

In [ ]:
CFG = REPO / "configs" / "glaucoma.yaml"
overrides = [
    f"run_name={RUN}",
    "model.architecture=pyramid3dcnn",
    "model.fusion_dim=64",
    "model.fuse_levels=3",
    f"data.data_dir={DATA_DIR}",
    "data.normalize=minmax",
    "logging.wandb=true",
    "logging.tensorboard=true",
    f"output_dir={OUT_DIR}",
]
cmd = [sys.executable, "-m", "pipeline.train", "--cfg", str(CFG)]
if SMOKE:
    cmd += ["--smoke"]
    overrides += ["train.epochs=1", "train.eval_every_steps=2", "train.eval_test_every_steps=4"]
else:
    overrides += [
        "train.epochs=15",
        "train.batch_size=4",
        "train.grad_accum_steps=2",
        "train.lr=0.0001",
        "train.eval_every_steps=100",
        "train.eval_test_every_steps=300",
        "train.save_every_steps=300",
        "train.early_stop_patience=6",
    ]
cmd += ["--set"] + overrides
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

run_dir = OUT_DIR / RUN
metrics = pd.read_json(run_dir / "metrics.jsonl", lines=True)
fig_dir = WORK / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for split in ("train", "val", "test"):
    for ax, key in zip(axes, ("loss", "acc", "f1")):
        col = f"{split}/{key}"
        if col in metrics:
            ax.plot(metrics["step"], metrics[col], label=split)
for ax, title in zip(axes, ("loss", "accuracy", "F1")):
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=0.3); ax.legend()
png = fig_dir / "pyramid_curves.png"
fig.tight_layout(); fig.savefig(png, dpi=150)
test = json.loads((run_dir / "test_results.json").read_text()) if (run_dir / "test_results.json").exists() else {}
print(json.dumps(test, indent=2))

In [ ]:
import wandb
wandb.init(project="glaucoma-thesis", id=RUN, resume="allow")
art = wandb.Artifact(f"{RUN}-results", type="model")
for name in ("best_model.pt", "metrics.jsonl", "config_used.json", "test_results.json", "training_curves.png"):
    p = run_dir / name if (run_dir / name).exists() else fig_dir / name
    if p.exists():
        art.add_file(str(p))
wandb.log_artifact(art)
wandb.finish()
print("artifact logged:", [f for f in ("best_model.pt", "metrics.jsonl", "test_results.json") if (run_dir / f).exists()])

## After the run
* Pipeline logged train/val/test loss, acc, precision, recall, F1 + `sys/*` live to W&B
  `glaucoma-thesis`; the last cell uploads best model, metrics, config and curves as an artifact.
* **Save Version** so `/kaggle/working` persists. Do not rely on the interactive session.
* Read balanced accuracy / MCC, not raw accuracy alone (test class counts 411/489).
* Ablations (same notebook): `model.fuse_levels=0` (no top-down fusion), `=2` (less VRAM),
  `model.fusion_dim=32`, and the 96^3 bilateral input (`tqhuyen/harvard-oct-glaucoma-200-bilateral-96`).
* GPU cost is logged by W&B; report sec/step and peak VRAM next to the baseline in
  `docs/pyramid-cnn.md`.